In [1]:
import torch
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}")
print(f"VRAM: {gpu.total_memory / 1024**3:.1f} GB")

GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/moc_results', exist_ok=True)
print("Drive mounted. Results will be saved to /content/drive/MyDrive/moc_results/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Results will be saved to /content/drive/MyDrive/moc_results/


In [3]:
import os

if os.path.exists('/content/dvlm-project'):
    %cd /content/dvlm-project
    !git pull
else:
    !git clone https://github.com/ShumailaJaved/dvlm-project.git /content/dvlm-project
    %cd /content/dvlm-project

print("Working directory:", os.getcwd())
!ls

/content/dvlm-project
Already up to date.
Working directory: /content/dvlm-project
checkpoints  eval     LLaVA_ScienceQA_baseline.ipynb  patch3.ps1
connector    figures  losses			      results
data	     LLaVA    paper			      setup_qlora.py


In [4]:
!pip install -q --upgrade torchao
import importlib, torchao
importlib.reload(torchao)
print(f"torchao version: {torchao.__version__}")

torchao version: 0.17.0


In [5]:
!pip install -q "bitsandbytes>=0.41" "peft>=0.9" accelerate "transformers>=4.37" datasets einops tqdm
print("Done.")

Done.


In [6]:
!pip install -q --no-deps -e ./LLaVA
print("LLaVA installed (no-deps). Checking torch version:")
import torch
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llava (pyproject.toml) ... done
LLaVA installed (no-deps). Checking torch version:
torch: 2.11.0+cu128
CUDA available: True


In [7]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.chdir('/content/dvlm-project')

In [8]:
!python setup_qlora.py

CHECK 1: Loading LLaVA-1.5-7B in 4-bit NF4 …
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
GPU memory before loading model : 0.00 GB
Loading liuhaotian/llava-v1.5-7b in 4-bit NF4 …
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
Fetching 2 files: 100% 2/2 [00:00<00:00, 35696.20it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights:   1% 4/295 [00:00<00:06, 42.69it/s, Materializing param=model.layers.0.mlp.down_proj.weight]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning:

In [9]:
!python connector/question_pooler.py

Verification device: cuda, dtype: torch.float16

CHECK 1 PASSED  (|q1 - q2|_2 = 33.0000 > 0)
CHECK 2 PASSED  (|q1 - q3|_2 = 0.0 — deterministic)
CHECK 3 PASSED  (shape: torch.Size([4096]))

ALL CHECKS PASSED


In [10]:
!python connector/expert_e1.py
!python connector/expert_e2.py
!python connector/expert_e3.py
!python connector/expert_e4.py

CHECK 1 PASSED  (output shape: torch.Size([576, 4096]))
CHECK 2 PASSED  (trainable parameters: 0)

ALL CHECKS PASSED
CHECK 1 PASSED  (output shape: torch.Size([32, 4096]))
CHECK 2 PASSED  (max attention weight row-sum deviation: 1.19e-07 < 0.0001)
CHECK 3 PASSED  (trainable parameters: 8,519,680 = 32×4096 + 2×1024×4096)

ALL CHECKS PASSED
CHECK 1 PASSED  (output shape: torch.Size([1, 4096]))
CHECK 2 PASSED  (attention weights sum to 1: image1=1.000000, image2=1.000000)
CHECK 2 INFO    (peak patch index: image1=427, image2=405 — shifts=yes)
CHECK 3 PASSED  (trainable parameters: 4,199,424 = 1024 + 1024×4096 + 4096)

ALL CHECKS PASSED
CHECK 1 PASSED  (output shape: torch.Size([576, 4096]))
CHECK 2 PASSED  (all gate values in (0, 1); min=0.5000, max=0.5000)
CHECK 3 PASSED  (α sums to 1.00000000 ≈ 1.0; deviation < 1e-05)
CHECK 4 PASSED  (tau_g = 1.0000 > 0)
CHECK 5 PASSED  (|G1 − G2|_F = 0.0003 > 0 — gates differ between questions)

Parameter count: 5,511,169 (expected 5,511,169)

ALL CHEC

In [11]:
!python connector/router.py

Verification device: cuda

CHECK 1 PASSED  (r at init ≈ [0.25, 0.25, 0.25, 0.25], max dev = 0.00e+00)
CHECK 2 PASSED  (W2.weight.grad is non-zero; |grad|_sum = 11.784094)
CHECK 3 PASSED  (k_star = 0 ∈ {0, 1, 2, 3})

ALL CHECKS PASSED


In [12]:
!python losses/load_balance.py

Verification device: cuda

CHECK 1 PASSED  (perfectly balanced: L_lb = 1.0000 ≈ 1.0)
CHECK 2 PASSED  (fully collapsed: L_lb = 0.7675 = K * p_0 = 4 * 0.1919 = 0.7675 ≤ 4)
CHECK 3 PASSED  (f_k.requires_grad = False = False ✓, p_k.requires_grad = True = True ✓)

ALL CHECKS PASSED


In [22]:
!git pull

remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.72 KiB | 879.00 KiB/s, done.
From https://github.com/ShumailaJaved/dvlm-project
   84dd53b..916fea5  main       -> origin/main
   68dfa2d..d17dae9  fixbug/moc -> origin/fixbug/moc
Updating 84dd53b..916fea5
Fast-forward
 connector/moc.py | 19 ++++++++++++++++++-
 1 file changed, 18 insertions(+), 1 deletion(-)


In [23]:
!python connector/moc.py --full

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Verification device: cuda

CHECK 1 PASSED  (k_star=0, V shape=torch.Size([576, 4096]), r.sum()=1.000000 ≈ 1.0)
CHECK 2 PASSED  (question changes cause different routing: experts selected = [0, 1, 2, 3])

Running full model checks (Colab A100) …
Loading LLaVA-1.5-7B in 4-bit …
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
Fetching 2 files: 100% 2/2 [00:00<00:00, 38130.04it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights:   1% 4/295 [00:00<00:06, 42.97it/s, Materia

In [24]:
import shutil, os

src = '/content/dvlm-project'
dst = '/content/drive/MyDrive/moc_results/verification_outputs'
os.makedirs(dst, exist_ok=True)

# Copy results and figures if any were generated
for folder in ['results', 'figures']:
    folder_path = os.path.join(src, folder)
    if os.path.exists(folder_path):
        shutil.copytree(folder_path, os.path.join(dst, folder), dirs_exist_ok=True)

print("Done. Check /content/drive/MyDrive/moc_results/verification_outputs/")

Done. Check /content/drive/MyDrive/moc_results/verification_outputs/
